# 23 — Prompt Observability and Failure Diagnosis

    ## Scenario and success criteria

    A wrong price answer must be traced to stale evidence without logging customer email addresses or hidden reasoning.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Capture correlated structured spans.
- Redact sensitive attributes.
- Diagnose the first observable failing layer.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 23 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

A complete transcript can leak data while still omitting the version and freshness fields needed for diagnosis.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab23 import diagnose, safe_span

healthy = [
    safe_span("trace-1", "retrieve", "ok", 18, {"query": "user@example.com", "evidence_fresh": True, "source_version": "v2"}),
    safe_span("trace-1", "generate", "ok", 70, {"schema_valid": True, "model": "fixture"}),
]
stale = [
    safe_span("trace-2", "retrieve", "ok", 17, {"query": "user@example.com", "evidence_fresh": False, "source_version": "v1"}),
    safe_span("trace-2", "generate", "ok", 68, {"schema_valid": True, "model": "fixture"}),
]

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
print("healthy diagnosis", diagnose(healthy))
print("stale diagnosis", diagnose(stale))
print("redacted query", stale[0].attributes["query"])

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert diagnose(healthy) == "no_observed_failure"
assert diagnose(stale) == "stale_evidence"
assert stale[0].attributes["query"] == "[REDACTED_EMAIL]"

## Production upgrade

Emit trace IDs, artifact/model/tool versions, policy results, evidence IDs, latency, usage, errors, and terminal state. Minimize content capture, set retention controls, and align names with OpenTelemetry conventions where stable.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.